# Manual multilabel training

The epoch loop is kept here for interactive inspection. Reusable operations live in `src/`.

In [1]:
from copy import deepcopy
from pathlib import Path
import sys

import pandas as pd
import torch
import yaml
from torch.utils.data import DataLoader

In [2]:
ROOT = Path.cwd().resolve()
if not (ROOT / "configs/framework.yaml").exists():
    ROOT = ROOT.parent
if not (ROOT / "configs/framework.yaml").exists():
    raise FileNotFoundError("Run this notebook from the repository or notebooks directory")
sys.path.insert(0, str(ROOT))

from src.data import fit_standardizer, make_dataset, split_indices
from src.hashing import calculate_run_id
from src.model import build_model
from src.training import (
    build_loss,
    build_optimizer,
    evaluate_epoch,
    save_artifacts,
    train_epoch,
)

In [3]:
framework_path = ROOT / "configs/framework.yaml"
framework_snapshot = framework_path.read_bytes()
config = yaml.safe_load(framework_snapshot)
project_path = ROOT / config["project"]["config_path"]
project_snapshot = project_path.read_bytes()
project_config = yaml.safe_load(project_snapshot)
config["project"].update(project_config)
project_basename = project_path.stem
run_id = calculate_run_id(config)
print(f"project: {project_basename}, run_id: {run_id}")

project: bmra_student_project, run_id: b5874d285eb0


In [4]:
data_path = ROOT / config["data"]["path"]
frame = pd.read_parquet(data_path)
feature_columns = project_config["features"]
label_columns = project_config["labels"]
missing_columns = set(feature_columns + label_columns) - set(frame.columns)
if missing_columns:
    raise ValueError(f"Columns not found in dataset: {sorted(missing_columns)}")
frame[feature_columns + label_columns].head()

,Abdominal Hernia,Ankle Dorsiflexion Deficit,Ankle Edema,Ankle Instability,Ankle Sprain History,Ankle Supination Deficit,Antalgic Posture,Aphthous Ulcers,Apley Inferior Test Deficit,Arm Injury,...,ALA,ALCAR,NAC,Vitamin C,Whey,Vegan protein,Multivitamin,Insoles correction,Supplements links,Good shoes
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
split = config["split"]
train_idx, val_idx, test_idx = split_indices(
    len(frame), split["train"], split["val"], split["test"], split["seed"]
)
if min(map(len, (train_idx, val_idx, test_idx))) == 0:
    raise ValueError("Dataset is too small for non-empty train, validation, and test splits")

means, scales = fit_standardizer(frame.iloc[train_idx], feature_columns)
train_data = make_dataset(frame.iloc[train_idx], feature_columns, label_columns, means, scales)
val_data = make_dataset(frame.iloc[val_idx], feature_columns, label_columns, means, scales)
test_data = make_dataset(frame.iloc[test_idx], feature_columns, label_columns, means, scales)

batch_size = config["training"]["batch_size"]
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size)
test_loader = DataLoader(test_data, batch_size=batch_size)
len(train_data), len(val_data), len(test_data)

(1075, 230, 232)

In [6]:
torch.manual_seed(split["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model(config, project_config).to(device)
criterion = build_loss(config["loss"]).to(device)
optimizer = build_optimizer(model, config["training"])
print(model)
print(f"device: {device}, loss: {criterion.__class__.__name__}")

MultilabelMLP(
  (network): Sequential(
    (0): Linear(in_features=614, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=128, out_features=64, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.2, inplace=False)
    (9): Linear(in_features=64, out_features=76, bias=True)
  )
)
device: cpu, loss: MaskedBCELoss


Run the next cell to train. Interrupt between epochs if you want to inspect `model` or a batch manually.

In [7]:
patience = config["training"]["early_stopping_patience"]
if patience < 1:
    raise ValueError("training.early_stopping_patience must be at least 1")

history = []
best_val_loss = float("inf")
best_epoch = 0
best_state = None
epochs_without_improvement = 0
stopped_early = False
for epoch in range(1, config["training"]["epochs"] + 1):
    train_metrics = train_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics = evaluate_epoch(model, val_loader, criterion, device)
    row = {
        "epoch": epoch,
        **{f"train_{key}": value for key, value in train_metrics.items()},
        **{f"val_{key}": value for key, value in val_metrics.items()},
    }
    history.append(row)

    if best_state is None or val_metrics["loss"] < best_val_loss:
        best_val_loss = val_metrics["loss"]
        best_epoch = epoch
        best_state = deepcopy(model.state_dict())
        best_train_metrics = train_metrics.copy()
        best_val_metrics = val_metrics.copy()
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(
        f"{epoch:03d} | train loss {train_metrics['loss']:.4f} "
        f"f1 {train_metrics['micro_f1']:.3f} | val loss {val_metrics['loss']:.4f} "
        f"f1 {val_metrics['micro_f1']:.3f} acc {val_metrics['accuracy']:.3f} "
        f"| patience {epochs_without_improvement}/{patience}"
    )
    if epochs_without_improvement >= patience:
        stopped_early = True
        print(f"Early stopping at epoch {epoch}; best epoch was {best_epoch}")
        break

model.load_state_dict(best_state)

001 | train loss 0.6621 f1 0.079 | val loss 0.5794 f1 0.067 acc 0.884 | patience 0/10
002 | train loss 0.3550 f1 0.060 | val loss 0.1590 f1 0.000 acc 0.957 | patience 0/10
003 | train loss 0.1734 f1 0.159 | val loss 0.1317 f1 0.053 acc 0.957 | patience 0/10
004 | train loss 0.1491 f1 0.129 | val loss 0.1278 f1 0.000 acc 0.957 | patience 0/10
005 | train loss 0.1430 f1 0.124 | val loss 0.1268 f1 0.000 acc 0.957 | patience 0/10
006 | train loss 0.1418 f1 0.115 | val loss 0.1258 f1 0.000 acc 0.957 | patience 0/10
007 | train loss 0.1410 f1 0.069 | val loss 0.1260 f1 0.003 acc 0.957 | patience 1/10
008 | train loss 0.1398 f1 0.109 | val loss 0.1261 f1 0.005 acc 0.957 | patience 2/10
009 | train loss 0.1383 f1 0.083 | val loss 0.1258 f1 0.005 acc 0.957 | patience 3/10
010 | train loss 0.1369 f1 0.095 | val loss 0.1252 f1 0.039 acc 0.957 | patience 0/10
011 | train loss 0.1364 f1 0.122 | val loss 0.1254 f1 0.034 acc 0.957 | patience 1/10
012 | train loss 0.1349 f1 0.124 | val loss 0.1249 f1 

<All keys matched successfully>

In [8]:
test_metrics = evaluate_epoch(model, test_loader, criterion, device)
metrics = {
    "project": project_basename,
    "run_id": run_id,
    "best_epoch": best_epoch,
    "epochs_trained": len(history),
    "stopped_early": stopped_early,
    "final_train": best_train_metrics,
    "final_validation": best_val_metrics,
    "test": test_metrics,
}
output_dir = save_artifacts(
    run_id,
    project_basename,
    model,
    framework_snapshot,
    project_snapshot,
    metrics,
    history,
    ROOT / "artifacts",
)
print(f"Saved artifacts to {output_dir}")
metrics

Saved artifacts to /media/chnguyen47/ADATA/Projects/BMRA_modular_MLC/artifacts/bmra_student_project/b5874d285eb0


{'project': 'bmra_student_project',
 'run_id': 'b5874d285eb0',
 'best_epoch': 15,
 'epochs_trained': 25,
 'stopped_early': True,
 'final_train': {'accuracy': 0.9536352509179926,
  'micro_f1': 0.21929101401483925,
  'loss': 0.13061557656803796},
 'final_validation': {'accuracy': 0.9570938215102975,
  'micro_f1': 0.09420289855072464,
  'loss': 0.12438332516214122},
 'test': {'accuracy': 0.9506011796733213,
  'micro_f1': 0.0821917808219178,
  'loss': 0.13916496521440044}}